# R8 · Warm-up: from the tree to the forest

**Type**: Implement with scikit-learn · **Level**: ●○○ guided · **Time**: about 25 minutes · **core path ★**

## Objective

Compare in cross-validation a single tree and a forest of 200 trees, check that the **out-of-bag** error estimates the test one well and read the three most important variables.

## Dataset

`load_breast_cancer`: 569 tumors with 30 measurements; `y = 1` if malignant. Stratified 5-fold cross-validation on all the data (`cv`), and a 70/30 split (`X_tr, X_te, y_tr, y_te`).

## Educational purpose

See the gain of bagging first hand (average of many trees ⇒ less variance), learn that the OOB is a free validation and that impurity importances have to be read with caution.

## Task

1. `tree_acc` and `forest_acc`: mean cross-validation accuracy (5 folds, `cv`) of `DecisionTreeClassifier(random_state=0)` and `RandomForestClassifier(200, random_state=0, n_jobs=-1)`, with the standard deviations `tree_sd`, `forest_sd`; `gain` is the difference between the means.
2. OOB: `forest = RandomForestClassifier(300, oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)`; save `oob` (`forest.oob_score_`) and `test_acc` (accuracy on the test set).
3. `top3`: the three variables with the largest impurity importance (`forest.feature_importances_`), in descending order.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer(as_frame=True)
X, y = data.data, (data.target == 0).astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

## Your work

In [ ]:
tree_scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=cv)
forest_scores = cross_val_score(..., X, y, cv=cv)
tree_acc, tree_sd = tree_scores.mean(), tree_scores.std()
forest_acc, forest_sd = ..., ...
gain = ...

In [ ]:
forest = RandomForestClassifier(300, oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
oob = ...
test_acc = ...

In [ ]:
top3 = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False).index[:3].tolist()`.'):
    check(0.90 < tree_acc < 0.95 and 0.95 < forest_acc < 0.99, "tree between 0.90 and 0.95, forest between 0.95 and 0.99", "cross_val_score(..., cv=cv).mean()")
    check(gain > 0.02 and forest_sd < tree_sd + 0.01, "the forest gains over 2 points and is not more variable than the tree", "forest_acc - tree_acc")
    check(abs(oob - test_acc) < 0.03 and oob > 0.94, "the OOB accuracy (over 0.94) is within 3 points of the test one: a free validation", "forest.oob_score_")
    check(len(top3) == 3 and sum("concave points" in v for v in top3) == 2, "among the three most important variables appear the two `concave points` measurements (`mean` and `worst`)", "np.argsort(-forest.feature_importances_)[:3]")
done()

---
The commented solution is in `solutions/B_with_sklearn/R8_warmup_from_the_tree_to_the_forest.ipynb`. Open it after you pass the check, or after you have really tried.